In [1]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6960.6/8062.4 GB disk)


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="M2hZer8PzPBpFRWbCD3q")
project = rf.workspace("madhusudans-workspace-31zqm").project("cc_ai_model-v1")
version = project.version(2)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CC_AI_Model-V1-2 in yolov8:: 100%|██████████| 78959/78959 [00:08<00:00, 9049.05it/s] 


In [3]:
import yaml, os

yaml_path = f"{dataset.location}/data.yaml"

# Check structure
print(os.listdir(dataset.location))

# Fix to absolute paths (prevents broken path errors mid-training)
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

config['train'] = f"{dataset.location}/train/images"
config['val']   = f"{dataset.location}/valid/images"
config['test']  = f"{dataset.location}/test/images"

with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

# Confirm what you're working with
print(f"Classes: {config['nc']} — {config['names']}")
print(f"Train images: {len(os.listdir(config['train']))}")

['test', 'valid', 'data.yaml', 'README.roboflow.txt', 'train', 'README.dataset.txt']
Classes: 6 — ['fire', 'intruder', 'liquid_spill', 'person', 'smoke', 'weapon']
Train images: 34542


In [4]:
import torch

print(f"CUDA devices: {torch.cuda.device_count()}")   # should be 2 on T4 x2
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Checkpoint exists: {os.path.exists('/kaggle/input/datasets/madhusudanambulkar/cloud-computing-pt-file/last.pt')}")

CUDA devices: 2
GPU: Tesla T4
Checkpoint exists: True


In [5]:
# Run this in a separate cell first
import os
print(os.path.exists('/kaggle/input/datasets/madhusudanambulkar/best-file/best_collab.pt'))  # must be True
print(dataset.location)  # must not throw NameError

True
/kaggle/working/CC_AI_Model-V1-2


In [6]:
from ultralytics import YOLO  # ← this line is missing

model = YOLO('/kaggle/input/datasets/madhusudanambulkar/best-file/best_collab.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=64,
    device=[0, 1],
    project='/kaggle/working',
    name='cc_ai_retrain',
    resume=False,
    patience=15,
    save=True,
    save_period=5,
    workers=4,
    plots=True,
    lr0=0.001,
    lrf=0.0001,
)

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/CC_AI_Model-V1-2/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.0001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/madhusudanambulkar/best-file/best_collab.pt

In [8]:
import shutil

shutil.copy(
    '/kaggle/working/cc_ai_retrain/weights/best.pt',
    '/kaggle/working/best_v4_final.pt'
)
print("Done ✅")

Done ✅
